In [ ]:
import csv
import os
from datetime import datetime


MENU_FILE = "restaurant_menu.csv"
ORDERS_FILE = "order_history.csv"


class MenuItem:
    """Represents one item in the restaurant menu."""

    def __init__(self, item_id, name, category, price, available=True):
        self.item_id = int(item_id)
        self.name = name
        self.category = category
        self.price = float(price)
        self.available = available

    def display(self):
        status = "Available" if self.available else "Unavailable"
        return f"{self.item_id:<5} {self.name:<25} {self.category:<15} ₹{self.price:>8.2f}  {status}"


class CartItem:
    """Represents an item and its quantity in the shopping cart."""

    def __init__(self, menu_item, quantity):
        self.menu_item = menu_item
        self.quantity = quantity

    def subtotal(self):
        return self.menu_item.price * self.quantity


class Restaurant:
    """Main controller class for the Restaurant Ordering System."""

    TAX_RATE = 0.05

    # Tuple demonstrates immutable fixed coupon definitions.
    COUPONS = ("WELCOME10", "SAVE20", "STUDENT15")

    def __init__(self):
        self.menu = {}
        self.cart = []
        self.used_coupons = set()
        self.load_menu()

    # ---------- FILE HANDLING ----------

    def create_default_menu_file(self):
        """Creates a CSV menu file if it does not exist."""
        if os.path.exists(MENU_FILE):
            return

        default_menu = [
            [1, "Veg Burger", "Fast Food", 120, "Yes"],
            [2, "Chicken Burger", "Fast Food", 160, "Yes"],
            [3, "Margherita Pizza", "Pizza", 220, "Yes"],
            [4, "Chicken Pizza", "Pizza", 280, "Yes"],
            [5, "French Fries", "Sides", 90, "Yes"],
            [6, "Paneer Wrap", "Wraps", 140, "Yes"],
            [7, "Chicken Wrap", "Wraps", 170, "Yes"],
            [8, "Veg Biryani", "Main Course", 180, "Yes"],
            [9, "Chicken Biryani", "Main Course", 240, "Yes"],
            [10, "Chocolate Milkshake", "Beverages", 110, "Yes"],
            [11, "Fresh Lime Soda", "Beverages", 70, "Yes"],
            [12, "Ice Cream", "Dessert", 80, "Yes"],
        ]

        with open(MENU_FILE, "w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(["id", "name", "category", "price", "available"])
            writer.writerows(default_menu)

    def load_menu(self):
        """Loads menu items from the CSV file into a dictionary."""
        self.create_default_menu_file()

        try:
            with open(MENU_FILE, "r", newline="", encoding="utf-8") as file:
                reader = csv.DictReader(file)

                for row in reader:
                    item = MenuItem(
                        row["id"],
                        row["name"],
                        row["category"],
                        row["price"],
                        row["available"].lower() == "yes",
                    )
                    self.menu[item.item_id] = item

        except (FileNotFoundError, KeyError, ValueError) as error:
            print(f"Error loading menu: {error}")

    def save_order(self, customer_name, payment_method, subtotal,
                   discount, tax, total):
        """Saves completed order information to CSV."""
        file_exists = os.path.exists(ORDERS_FILE)

        with open(ORDERS_FILE, "a", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)

            if not file_exists:
                writer.writerow([
                    "date_time",
                    "customer_name",
                    "items",
                    "payment_method",
                    "subtotal",
                    "discount",
                    "tax",
                    "total",
                ])

            item_summary = "; ".join(
                f"{item.menu_item.name} x{item.quantity}"
                for item in self.cart
            )

            writer.writerow([
                datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                customer_name,
                item_summary,
                payment_method,
                f"{subtotal:.2f}",
                f"{discount:.2f}",
                f"{tax:.2f}",
                f"{total:.2f}",
            ])

    # ---------- DISPLAY METHODS ----------

    def show_menu(self):
        """Displays all available menu items."""
        print("\n" + "=" * 80)
        print("RESTAURANT MENU")
        print("=" * 80)
        print(f"{'ID':<5} {'Name':<25} {'Category':<15} {'Price':>10}  Status")
        print("-" * 80)

        for item in self.menu.values():
            print(item.display())

        print("=" * 80)

    def show_categories(self):
        """Displays unique menu categories using a set."""
        categories = {item.category for item in self.menu.values()}

        print("\nAvailable Categories:")
        for category in sorted(categories):
            print(f"- {category}")

    def search_menu(self):
        """Searches menu by item name or category."""
        keyword = input("Enter item name/category to search: ").strip().lower()

        results = [
            item for item in self.menu.values()
            if keyword in item.name.lower() or keyword in item.category.lower()
        ]

        if not results:
            print("No matching items found.")
            return

        print("\nSearch Results")
        print("-" * 80)
        for item in results:
            print(item.display())

    # ---------- CART METHODS ----------

    def find_cart_item(self, item_id):
        """Returns a cart item by menu ID."""
        for cart_item in self.cart:
            if cart_item.menu_item.item_id == item_id:
                return cart_item
        return None

    def add_to_cart(self):
        try:
            item_id = int(input("Enter item ID: "))
            quantity = int(input("Enter quantity: "))

            if quantity <= 0:
                print("Quantity must be greater than zero.")
                return

            if item_id not in self.menu:
                print("Invalid item ID.")
                return

            menu_item = self.menu[item_id]

            if not menu_item.available:
                print("Sorry, this item is currently unavailable.")
                return

            existing = self.find_cart_item(item_id)

            if existing:
                existing.quantity += quantity
            else:
                self.cart.append(CartItem(menu_item, quantity))

            print(f"{menu_item.name} added to cart successfully.")

        except ValueError:
            print("Invalid input. Please enter numbers only.")

    def remove_from_cart(self):
        if not self.cart:
            print("Your cart is empty.")
            return

        self.view_cart()

        try:
            item_id = int(input("Enter item ID to remove: "))
            cart_item = self.find_cart_item(item_id)

            if cart_item is None:
                print("Item is not in your cart.")
                return

            self.cart.remove(cart_item)
            print("Item removed successfully.")

        except ValueError:
            print("Invalid item ID.")

    def update_cart_quantity(self):
        if not self.cart:
            print("Your cart is empty.")
            return

        self.view_cart()

        try:
            item_id = int(input("Enter item ID: "))
            quantity = int(input("Enter new quantity: "))

            cart_item = self.find_cart_item(item_id)

            if cart_item is None:
                print("Item is not in your cart.")
                return

            if quantity <= 0:
                self.cart.remove(cart_item)
                print("Item removed from cart.")
            else:
                cart_item.quantity = quantity
                print("Quantity updated.")

        except ValueError:
            print("Please enter valid numbers.")

    def view_cart(self):
        if not self.cart:
            print("\nYour cart is empty.")
            return

        print("\n" + "=" * 70)
        print("YOUR CART")
        print("=" * 70)
        print(f"{'ID':<5} {'Item':<25} {'Qty':<8} {'Price':<12} {'Subtotal'}")
        print("-" * 70)

        for cart_item in self.cart:
            print(
                f"{cart_item.menu_item.item_id:<5} "
                f"{cart_item.menu_item.name:<25} "
                f"{cart_item.quantity:<8} "
                f"₹{cart_item.menu_item.price:<10.2f} "
                f"₹{cart_item.subtotal():.2f}"
            )

        print("-" * 70)
        print(f"{'Cart Total':>55}: ₹{self.calculate_subtotal():.2f}")

    def calculate_subtotal(self):
        return sum(item.subtotal() for item in self.cart)

    def clear_cart(self):
        if not self.cart:
            print("Cart is already empty.")
            return

        confirm = input("Clear the entire cart? (y/n): ").strip().lower()

        if confirm == "y":
            self.cart.clear()
            self.used_coupons.clear()
            print("Cart cleared.")
        else:
            print("Operation cancelled.")

    # ---------- BILLING ----------

    def apply_coupon(self, subtotal):
        """Applies one of the available coupon codes."""
        if subtotal <= 0:
            return 0

        print("\nAvailable Coupons:")
        print("WELCOME10  -> 10% discount")
        print("SAVE20     -> 20% discount on orders >= ₹500")
        print("STUDENT15  -> 15% discount on orders >= ₹300")

        coupon = input("Enter coupon code (or press Enter to skip): ").strip().upper()

        if not coupon:
            return 0

        if coupon not in self.COUPONS:
            print("Invalid coupon code.")
            return 0

        if coupon in self.used_coupons:
            print("Coupon has already been used for this cart.")
            return 0

        discount_rate = 0

        if coupon == "WELCOME10":
            discount_rate = 0.10

        elif coupon == "SAVE20":
            if subtotal >= 500:
                discount_rate = 0.20
            else:
                print("SAVE20 requires a minimum order of ₹500.")

        elif coupon == "STUDENT15":
            if subtotal >= 300:
                discount_rate = 0.15
            else:
                print("STUDENT15 requires a minimum order of ₹300.")

        if discount_rate > 0:
            discount = subtotal * discount_rate
            self.used_coupons.add(coupon)
            print(f"Coupon applied! You saved ₹{discount:.2f}.")
            return discount

        return 0

    def checkout(self):
        """Generates bill, accepts payment method, and saves the order."""
        if not self.cart:
            print("Cannot checkout because the cart is empty.")
            return

        customer_name = input("Enter customer name: ").strip()

        if not customer_name:
            customer_name = "Guest"

        subtotal = self.calculate_subtotal()
        discount = self.apply_coupon(subtotal)
        taxable_amount = subtotal - discount
        tax = taxable_amount * self.TAX_RATE
        total = taxable_amount + tax

        print("\nPayment Methods:")
        print("1. Cash")
        print("2. UPI")
        print("3. Card")

        payment_options = {
            "1": "Cash",
            "2": "UPI",
            "3": "Card",
        }

        payment_choice = input("Select payment method: ").strip()

        if payment_choice not in payment_options:
            print("Invalid payment method. Checkout cancelled.")
            return

        payment_method = payment_options[payment_choice]

        print("\n" + "=" * 60)
        print("FINAL BILL")
        print("=" * 60)
        print(f"Customer       : {customer_name}")
        print(f"Date & Time    : {datetime.now().strftime('%d-%m-%Y %H:%M:%S')}")
        print("-" * 60)

        for cart_item in self.cart:
            print(
                f"{cart_item.menu_item.name:<30} "
                f"x{cart_item.quantity:<3} "
                f"₹{cart_item.subtotal():.2f}"
            )

        print("-" * 60)
        print(f"Subtotal       : ₹{subtotal:.2f}")
        print(f"Discount       : ₹{discount:.2f}")
        print(f"Tax (5%)       : ₹{tax:.2f}")
        print(f"Grand Total    : ₹{total:.2f}")
        print(f"Payment        : {payment_method}")
        print("=" * 60)
        print("Thank you for ordering!")

        self.save_order(
            customer_name,
            payment_method,
            subtotal,
            discount,
            tax,
            total,
        )

        self.cart.clear()
        self.used_coupons.clear()

    # ---------- ORDER HISTORY ----------

    def view_order_history(self):
        if not os.path.exists(ORDERS_FILE):
            print("No previous orders found.")
            return

        try:
            with open(ORDERS_FILE, "r", newline="", encoding="utf-8") as file:
                reader = csv.DictReader(file)
                rows = list(reader)

            if not rows:
                print("No previous orders found.")
                return

            print("\n" + "=" * 100)
            print("ORDER HISTORY")
            print("=" * 100)

            for index, row in enumerate(rows, start=1):
                print(f"\nOrder #{index}")
                print(f"Date       : {row['date_time']}")
                print(f"Customer   : {row['customer_name']}")
                print(f"Items      : {row['items']}")
                print(f"Payment    : {row['payment_method']}")
                print(f"Subtotal   : ₹{row['subtotal']}")
                print(f"Discount   : ₹{row['discount']}")
                print(f"Tax        : ₹{row['tax']}")
                print(f"Total      : ₹{row['total']}")

            print("=" * 100)

        except (FileNotFoundError, KeyError) as error:
            print(f"Could not read order history: {error}")

    # ---------- MAIN MENU ----------

    def run(self):
        while True:
            print("\n" + "=" * 60)
            print("      🍽️  RESTAURANT ORDERING SYSTEM")
            print("=" * 60)
            print("1. View Menu")
            print("2. View Categories")
            print("3. Search Menu")
            print("4. Add Item to Cart")
            print("5. View Cart")
            print("6. Update Cart Quantity")
            print("7. Remove Item from Cart")
            print("8. Clear Cart")
            print("9. Checkout")
            print("10. View Order History")
            print("11. Exit")
            print("=" * 60)

            choice = input("Enter your choice (1-11): ").strip()

            if choice == "1":
                self.show_menu()

            elif choice == "2":
                self.show_categories()

            elif choice == "3":
                self.search_menu()

            elif choice == "4":
                self.add_to_cart()

            elif choice == "5":
                self.view_cart()

            elif choice == "6":
                self.update_cart_quantity()

            elif choice == "7":
                self.remove_from_cart()

            elif choice == "8":
                self.clear_cart()

            elif choice == "9":
                self.checkout()

            elif choice == "10":
                self.view_order_history()

            elif choice == "11":
                print("\nThank you for using the Restaurant Ordering System!")
                print("Goodbye! 👋")
                break

            else:
                print("Invalid choice. Please select a number from 1 to 11.")


def main():
    """Program entry point."""
    restaurant = Restaurant()
    restaurant.run()


if __name__ == "__main__":
    main()



      🍽️  RESTAURANT ORDERING SYSTEM
1. View Menu
2. View Categories
3. Search Menu
4. Add Item to Cart
5. View Cart
6. Update Cart Quantity
7. Remove Item from Cart
8. Clear Cart
9. Checkout
10. View Order History
11. Exit
